# Day 025 · 主流交易所对比
**Major Exchanges** · 阶段 P1 · 量化基础

> 今天是第一阶段25 节课的收官,也是最接近散户日常体感的一节。前24 天我们建立了金融工程的核心理论 — 收益、风险、定价、对冲、订单簿、撮合、价差冲击。但这些理论在不同的交易所落地时,细节差别巨大。今天我们做横向对比,把全球几家主要交易所摆在一起,看清楚 A 股、美股、港股、加密这四类市场在交易时段、撮合规则、涨跌停、熔断、跨市场通道五个维度上的关键差别。每一条差别都对应一个散户最容易踩的坑 — 美股开盘那一刻的盘前数字陷阱、A 股涨停板的封单诱惑、沪深港通的额度限制、加密市场没有保护伞的裸奔风险、芝商所按比例分配跟美股先进先出的撮合差异。学完这一节,你的策略框架将具备跨市场配置能力,而不是把一套美股直觉硬套到 A 股或加密上去交学费。

---

**课件生成日期:** 2026-05-18  ·  **建议学习时长:** 18 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 看懂全球四类主流市场(美股 / A 股 / 港股 / 加密)的交易时段差别,以及24 小时全球接力的物理基础
- 理解 T 加一对比 T 加零、涨跌停对比限价熔断对比无熔断三种异常保护机制的本质差别
- 看清 A 股涨停板封单跟成交概率反相关这个反直觉现象,以及为什么追涨停板大概率被套
- 理解沪深港通通道的额度、汇率、假日不同步三大隐性成本,知道跨境通道的局限
- 区分先进先出跟按比例分配两种撮合规则,以及为什么把美股直觉搬到芝商所利率期货策略会失效
- 掌握跨市场策略的配置化思路 — 把市场参数做成字典,而不是硬编码在策略主体里

## 历史背景:全球24 小时交易接力 — 三类市场三种气质

你打开手机想下单的那一刻,你面对的不是一个抽象的市场,而是一类有具体性格的交易场所。我用三个生活场景给你打个比方,你立刻就能记住这三类市场的气质。

第一类美股,像咖啡店。早上9:30开门,下午四点关门,周末歇业。所有人挤在六个半小时里挑选自己的咖啡,流动性最集中,生意最热闹。关门时间一到必须结账,你不能赖着不走。这决定了美股策略基本是日内进出,持仓过夜的成本和风险都高。

第二类 A 股,像食堂。早上9:30开餐,11:30收摊午休,下午一点再开张,三点准时关门。一天四个小时窗口,中午一个半小时停盘是雷打不动的规矩。你想吃辣的得避开午休时段,日内回转更是没门 — 当天买的股票第2 个交易日才能卖。所以 A 股很多散户被迫做趋势,不是因为他们爱做,而是规则逼的。

第三类加密,像24 小时便利店。从来不关门,过年过节深更半夜都开着。听起来很自由,但你晚上十二点去便利店,你能买到的东西没白天多,服务员可能在打盹,出了事你也找不到人投诉。加密成交量分布极不均匀,北美下午到亚洲晚上是高峰,周末整体只有平时的两三成,黑天鹅事件经常专挑周末出。

这三种气质决定了散户在每个市场该用什么节奏 — 美股拼短打,A 股拼耐心,加密拼风控。学完这节课你会意识到,选对市场比选对策略更重要,因为市场的规则会自己塑造适合它的策略。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 交易时段 + 午休 — 四类市场的物理节奏完全不同

交易时段不是个枯燥的时间表,它是市场的生物钟,直接决定你的策略能不能跑、什么时段流动性最好、什么时段是黑天鹅高发期。

美股交易时段是美东时间早上9:30到下午四点,夏令时换算到北京时间是晚上9:30到第2 天凌晨四点,冬令时晚一小时。盘前从美东时间凌晨四点到9:30,盘后从下午四点到晚上八点,但这两段流动性大约只有日间的5%到10%,几个大单就能把价格推到不合理的位置。这就是为什么散户经常看到盘前一百九十买,9:30开盘跳到一百九十二的原因 — 盘前数字是少数几家做市商挂出来的,不是真实供需出清的价格。

A 股交易时段是北京时间早上9:30到下午三点,但中间有个一个半小时的午休,11:30到下午一点完全停盘。九点十五到九点二十五是开盘集合竞价,二十之后不能撤单是个特别坑散户的细节。下午两点五十七到三点是收盘集合竞价。一天有效连续撮合时段只有四个小时左右,这逼着 A 股的散户把策略压缩在这四个小时里完成。

港股交易时段是香港时间早上9:30到下午四点,中午十二点到一点休市一小时,比 A 股短半个小时。芝商所期货是周日晚上到周五下午几乎不停机,每天只在美东时间下午四点到五点停一小时给系统对账。加密市场则是真正的24 小时全年无休,但流动性集中在北美下午到亚洲晚上这段重叠时间。

实战含义有四点。第一,跨市场策略必须把时段差别做成配置,不能硬编码。第二,A 股中午一个半小时停盘期间你的仓位敞口纯靠隔夜对冲,这是 A 股策略最头疼的盲区。第三,加密做市必须全自动化,人工根本盯不住24 小时。第四,芝商所夜盘成交量大约是日间的5%,但能反映亚洲机构对美股的看法,A 股开盘经常跟夜盘期货走 — 这就是跨市场联动的桥梁。

> **举例:** 一个真实场景 — 2023 年某个周末加密市场比特币突然下跌15%,因为某家交易所传出提币风险。同一时刻 A 股、美股、港股全部休市,你想用沪深300 期货或者标普500 期货对冲都做不到,只能眼睁睁看着仓位裸奔到周一开盘。如果你的加密仓位占总资产超过20%,周末的纯敞口就足够让你失眠。


### 2. T 加一对比 T 加零 — A 股最大的独特规则

T 加一的意思是当天买进的股票第2 个交易日才能卖,字面上 T 是交易日 transaction date,加一是隔一天。A 股从1995 年开始就是 T 加一,几乎所有股票都受这条规则约束。美股、港股、加密、绝大多数期货都是 T 加零 — 当天买当天能卖,理论上你一天可以来回交易几十次。

打个比方,T 加一就像你在外卖平台下单买了一份蛋糕,平台规定你今天买的蛋糕明天才能吃。哪怕你下单后5 分钟反悔了,你也只能等到明天再退或者再处理。T 加零就是你在便利店买东西,买完转身就能退、就能再卖给下一个人。这两种规则塑造了完全不同的交易生态。

T 加一对散户的实战影响有五个层面。第一,日内回转策略在 A 股做不了,你不能上午买下午卖。第二,你的止损执行有滞后 — 上午买了之后下午跌了5%你想跑也跑不掉。第三,逼着 A 股散户做趋势,因为你只能持仓到第2 天才能卖,所以你必须判断隔夜消息和大盘方向。第四,科创板和创业板有交易型基金的 T 加零变体,比如部分宽基交易型基金可以日内回转,这是 A 股留给散户的少数 T 加零选择之一。第五,2024 年美股结算从 T 加二改成了 T 加一,但这里的 T 加一指的是结算日,不是交易限制 — 美股仍然可以日内任意次数交易,只是清算的钱第2 天到账,跟 A 股那种交易限制是两码事,别搞混。

反过来看 T 加零市场的特点。美股一天可以无限次日内交易,但有个隐形门槛叫日内交易规则,账户余额低于两万5000 美元的散户一周最多只能做三次日内交易,超过会被限制。这条规则就是为了防止散户被高频做市机器人收割得太惨。加密市场没有任何限制,你一天交易上百次都行,但杠杆放大下亏损也会成百 倍。

> **举例:** 一个真实的散户故事。某个新手2023 年从美股转过来做 A 股,他习惯了美股那种日内进出。在 A 股某只股票上他上午十点买了30000 股,十一点二十看到大盘走弱想跑,结果发现卖不了。中午午休那一个半小时他坐立不安。下午一点开盘股票直接低开3%,他想立刻平仓,但是这天他要么扛过夜,要么按低开价砸出去。这种被规则强行套牢的体验,是 A 股给所有跨市场转来散户的第一课。


### 3. 异常保护 — 涨跌停对比限价熔断对比无熔断

异常状态下三个市场的保护伞完全不同,这一条决定了你的策略风控架构。

A 股的保护机制叫涨跌停板,机制最严格。主板正负10%,创业板和科创板正负20%,特别处理的股票正负5%。一天最多涨10%、跌10%,理论上不会出现极端闪崩。但是涨停板的副作用是流动性瞬间消失 — 跌停板上可以挂卖单但没有买盘,你想止损止不掉,只能眼睁睁看着账户净值在跌停板上躺着。这就是 A 股散户最痛的体验之一。

美股的保护机制叫限价熔断,2010 年5 月闪崩之后强制引入。规则是5 分钟内股价偏离参考价5%到10%,自动暂停5 秒,具体阈值看股票分类。第一类大盘股的阈值是5%,其他股票是10%。触发后进入限价状态15 秒,如果价格反向回归就继续交易,否则暂停5 秒。优点是市场短暂冷却,不影响整体价格发现,缺点是碎片化的多交易所环境下可能反复触发。

港股2016 年也加了限价熔断,规则跟美股类似但参数不同 — 5 分钟内偏离参考价正负10%,暂停5 分钟。改革之前港股闪崩频繁,2015 年8 月24 日中环闪崩是个标志事件,改革后稳定不少。

芝商所期货有自己的熔断,以微型标普500 期货为例,跌幅触发三级阈值 — 7%、13%、20%。前两级各暂停15 分钟,20%当天直接停盘。这套机制是1987 年黑色星期一之后引入的。

加密市场没有任何强制熔断,这是它跟传统市场最大的区别。币安、Coinbase 这些主要交易所都没有断路器,撮合一直在跑,价格爱跌多少跌多少。2022 年5 月露娜币5 天跌幅99.99%不是个案,撮合引擎正常运行,价格一路向下,任何机制都不会保护你。所以加密策略必须自己实现熔断逻辑 — 单日亏损达到本金的5%自动停机,这是基础底线。

> **举例:** 一个真实场景。2015 年8 月24 日全球闪崩日。美股开盘5 分钟内限价熔断触发一千多次,但最终市场仍然跌了7%。同一天 A 股千 股跌停,跌停板上只有卖单完全没有买盘,散户全部被困。同期加密市场比特币24 小时跌幅约百分之十八,撮合一直在跑没有任何保护。三种异常状态下做策略的处理逻辑完全不同 — 美股要处理瞬间熔断挂单,A 股要处理连续涨跌停的流动性消失,加密要靠自己的代码实现停机保护。这是写跨市场策略最大的陷阱,也是这节课最核心的一条结构性认知。


### 4. 撮合规则 + 最小变动价位 — 自动化策略的两个基础参数

撮合规则决定你能不能成交,最小变动价位决定你的成交成本下限。这两个参数是写任何自动化策略前必须先确认的基础配置。

撮合规则方面,绝大多数市场用先进先出加价格优先这套 — A 股、美股、港股、加密都是。出价好的先成交,同一价位先挂的先成交。你想抢成交概率,要么挂更好的价格,要么挂得更早。高频做市机器人就是花天量算力抢同价队列的第一位。

但是芝商所部分品种用按比例分配,主要是利率期货、英镑期货、部分国债。同一价位来对手单了,不按挂单时间分,按各家挂单数量的比例分。你挂100 手,别人挂10000 手,新对手单进来,你只能分到大约百分之一。这一套规则下排队位置不重要,挂单数量重要。把美股那种抢内侧第一位的直觉搬到芝商所利率期货,你的策略会直接失效。

最小变动价位是交易所规定的最小报价单位。A 股全部统一零点零一元。美股普通股票零点零一美元,小于1 美元的低价股变成零点零零零一美元。港股最有意思,按价格分档 — 十港币以下零点零一,十到20 港币零点零五,二十到100 港币零点一,价格越高最小变动越大。芝商所微型标普期货零点二五点,黄金零点1 美元。

最小变动价位决定了价差的物理下限。比如腾讯港股股价大概三百块港币,最小变动零点零五,价差物理下限就是零点1 港币,折成基点差不多就是三十多个基点。茅台 A 股股价大概一千八百块,最小变动零点零一,价差物理下限零点零二元,折成基点大概一个基点出头。同样高流动性的票,港股的低价票物理下限比 A 股大几十倍。这不是流动性差,是规则差。

实战含义是两条。第一,学新市场前必须先确认它用哪种撮合规则,先进先出还是按比例,这决定你的排队位置策略和挂单数量策略。第二,最小变动价位决定可交易价差的下限,做小盘股或低价票要先把物理下限算清楚,避免误把规则限制当成流动性紧。

> **举例:** 一个真实策略迁移故事。某机构2022 年想把它在纳斯达克跑得很好的高频做市策略搬到芝商所利率期货。他们直接套用了纳斯达克那套抢内侧加一档零点零零零一美元的策略。结果到了芝商所利率期货上,他们挂的100 手单子在十万手的对手单进来的时候只分到一手,完全没法成交。后来重写了按比例分配的版本,改成大单挂多档,挂单总量提高到10000 手以上,策略才跑起来。两套规则差不只是参数差,是策略原理差。


### 5. 跨市场通道 — 沪深港通对比中概股对比加密交易所桥

跨市场通道是散户和机构在不同市场之间转移资金的桥梁。理解这些通道的限制对你做跨市场策略至关重要。

沪深港通是 A 股和港股之间的官方通道,2014 年上线。我给你打个高铁的比方 — 这就像一趟北京到香港的高铁,车厢里座位是有限的。北向就是内地资金买港股,每天总额度大概五百二十亿人民币,日内累计成交到了这个数,通道就关闭。南向是港股资金买 A 股,每天总额度大概四百二十亿人民币。2018 年之前是单日严格限制,之后改成只剩月度限额,日内基本不会用爆。

沪深港通对散户实战的影响有三个细节。第一,港股报价用港币,你买卖结算用人民币,中间的汇率换算交易所帮你做,但是汇率波动的成本得你自己吃。比如人民币兑港币从零点九兑一突然跳到零点八五兑一,你哪怕港股股票一动不动,折成人民币就少了五个百分点。第二,可买股票池是有限的,大概三千只,主要是沪深三百加上证三百八十成分股加上港股通过的合资格股票。冷门小盘股两边都买不到。第三,假日不同步 — 内地放假但香港正常开市的时候,通道关闭,你看着港股在走但动不了。同理香港放假内地开市也一样。

中概股美国预托证券是另一种通道。阿里巴巴、台积电、丰田这些都是预托证券。它跟港股二级上市股票之间存在套利空间,2020 年以来价差大幅波动,反映监管不确定性。这块对资金量要求高,散户能参与的空间小。

加密跨交易所桥完全是另一个画风。一个比特币在币安、Coinbase、欧易、Kraken 几十家交易所同时交易,你不需要任何审批就能在任何一家开户。正常时候不同交易所的价差小于万分之五,套利空间已经被高频机器人吃干净了。但是异常时候,比如某家交易所被监管打击限制提币,价差可以拉到5%甚至更多。2022 年某交易所一周破产期间,它家比特币比其他交易所便宜了10%,这时候你看似有套利机会,实际你的钱根本提不出来。

实战教训。第一,沪深港通是 A 股加港股套利的主要通道,但日额度月额度是真实的瓶颈,大单提前盯紧通道使用率。第二,中概股套利对资金量要求高,散户慎入。第三,加密跨交易所价差走宽不是套利信号,是异常事件的预警,看到价差拉宽到1%以上就要警觉某家交易所可能出问题。

> **举例:** 一个真实场景。某内地散户2023 年6 月想做沪深港通买腾讯,他在某个周一上午十点挂单,看到通道额度只剩20%不到。他犹豫了20 分钟,中午看通道额度已经用完,挂单全部失败。下午港股开盘他眼看腾讯走高3%完全动不了。这一笔他原本可以挣3%的机会,因为不懂额度监控完全错过。沪深港通额度用尽是真实的执行风险,不是理论上的限制。


## 实操:全球主流交易所核心维度静态对比 + 24 小时市场重叠图 + 最小变动物理下限基点

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_025_exchange_compare.py — 主流交易所横向对比:时段 / 撮合 / 最小变动 / 异常保护
# 不联网,纯静态字典 + matplotlib 可视化。Run All 即可看到所有结果
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd

# ============ 1. 主流市场核心维度数据字典 ============
# 字段:open/close 用北京时间小时(小数,如 9.5 = 9:30)
#       lunch_start/end 午休时段(None = 无午休)
#       match_rule = 撮合规则枚举 ('FIFO' / 'PRORATA')
#       tick_size = 最小变动价位(currency unit)
#       price_limit = 涨跌停限制(None = 无)
#       circuit_breaker = 熔断类型 ('LULD' / 'STATIC' / 'EXCHANGE_LEVEL' / None)
#       trading_24h = 是否 24 小时交易
markets = {
    'A 股(上交所 / 深交所)': {
        'open_bj': 9.5, 'close_bj': 15.0,
        'lunch_start': 11.5, 'lunch_end': 13.0,
        'match_rule': 'FIFO', 'tick_size': 0.01, 'tick_currency': 'CNY',
        'price_limit_pct': 10.0, 'circuit_breaker': None,
        'trading_24h': False, 't_plus': 1,
        'rep_stock_price': 1800.0, 'rep_name': '茅台 1800 元'
    },
    '美股(纽交所 / 纳斯达克)': {
        'open_bj': 22.5, 'close_bj': 29.0,  # 跨日:第二天 5:00
        'lunch_start': None, 'lunch_end': None,
        'match_rule': 'FIFO', 'tick_size': 0.01, 'tick_currency': 'USD',
        'price_limit_pct': None, 'circuit_breaker': 'LULD',
        'trading_24h': False, 't_plus': 1,
        'rep_stock_price': 190.0, 'rep_name': '苹果 190 美元'
    },
    '港股(港交所)': {
        'open_bj': 9.5, 'close_bj': 16.0,
        'lunch_start': 12.0, 'lunch_end': 13.0,
        'match_rule': 'FIFO', 'tick_size': 0.05, 'tick_currency': 'HKD',
        'price_limit_pct': None, 'circuit_breaker': 'STATIC',
        'trading_24h': False, 't_plus': 2,
        'rep_stock_price': 300.0, 'rep_name': '腾讯 300 港币'
    },
    '芝商所微型标普': {
        'open_bj': 7.0, 'close_bj': 30.0,  # 几乎全天
        'lunch_start': 5.0, 'lunch_end': 6.0,  # 美东 16:00-17:00 停一小时
        'match_rule': 'FIFO', 'tick_size': 0.25, 'tick_currency': 'INDEX',
        'price_limit_pct': None, 'circuit_breaker': 'EXCHANGE_LEVEL',
        'trading_24h': False, 't_plus': 1,
        'rep_stock_price': 4500.0, 'rep_name': '微型标普 4500 点'
    },
    '加密(币安 / Coinbase)': {
        'open_bj': 0.0, 'close_bj': 24.0,
        'lunch_start': None, 'lunch_end': None,
        'match_rule': 'FIFO', 'tick_size': 0.01, 'tick_currency': 'USDT',
        'price_limit_pct': None, 'circuit_breaker': None,
        'trading_24h': True, 't_plus': 0,
        'rep_stock_price': 60000.0, 'rep_name': 'BTC 60000 USDT'
    },
}

# ============ 2. 横向对比表(打印) ============
rows = []
for name, m in markets.items():
    open_h = int(m['open_bj']); open_min = int((m['open_bj'] % 1) * 60)
    close_bj = m['close_bj'] if m['close_bj'] <= 24 else m['close_bj'] - 24
    close_h = int(close_bj); close_min = int((close_bj % 1) * 60)
    session_str = f"{open_h:02d}:{open_min:02d} - {close_h:02d}:{close_min:02d}"
    if m['close_bj'] > 24:
        session_str += ' (+1d)'
    lunch_str = f"{m['lunch_start']:.1f}-{m['lunch_end']:.1f}" if m['lunch_start'] is not None else '无'
    pl_str = f"±{m['price_limit_pct']:.0f}%" if m['price_limit_pct'] is not None else '无'
    cb_str = m['circuit_breaker'] if m['circuit_breaker'] is not None else '无'
    rows.append([name, session_str, lunch_str, m['match_rule'], f"{m['tick_size']:.4f}", pl_str, cb_str, f"T+{m['t_plus']}", '是' if m['trading_24h'] else '否'])

df = pd.DataFrame(rows, columns=['市场', '北京时间时段', '午休', '撮合', '最小变动', '涨跌停', '熔断', '结算', '24h'])
print('='*100)
print('全球主流交易所核心维度对比')
print('='*100)
print(df.to_string(index=False))

# ============ 3. 二十四小时市场重叠图 ============
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e63946', '#1d3557', '#f4a261', '#2a9d8f', '#9d4edd']
y_pos = list(range(len(markets)))
for i, (name, m) in enumerate(markets.items()):
    open_h = m['open_bj']
    close_h = m['close_bj']
    color = colors[i % len(colors)]
    # 主时段(可能跨日)
    if close_h <= 24:
        ax.add_patch(patches.Rectangle((open_h, i-0.35), close_h - open_h, 0.7, color=color, alpha=0.65))
    else:
        ax.add_patch(patches.Rectangle((open_h, i-0.35), 24 - open_h, 0.7, color=color, alpha=0.65))
        ax.add_patch(patches.Rectangle((0, i-0.35), close_h - 24, 0.7, color=color, alpha=0.65))
    # 午休 — 白色覆盖
    if m['lunch_start'] is not None:
        ls, le = m['lunch_start'], m['lunch_end']
        ax.add_patch(patches.Rectangle((ls, i-0.35), le - ls, 0.7, color='white', alpha=1.0))
        ax.add_patch(patches.Rectangle((ls, i-0.35), le - ls, 0.7, fill=False, edgecolor=color, hatch='///', alpha=0.5))

ax.set_yticks(y_pos)
ax.set_yticklabels(list(markets.keys()), fontsize=10)
ax.set_xlim(0, 24)
ax.set_ylim(-0.6, len(markets) - 0.4)
ax.set_xticks(range(0, 25, 2))
ax.set_xlabel('北京时间(小时)', fontsize=11)
ax.set_title('全球主流交易所交易时段(北京时间)— 24 小时接力图', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
ax.axvline(9.5, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
ax.axvline(22.5, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
ax.text(9.5, len(markets)-0.2, 'A 股开盘', fontsize=8, color='red', ha='center')
ax.text(22.5, len(markets)-0.2, '美股开盘', fontsize=8, color='red', ha='center')
plt.tight_layout()
plt.savefig('chart_01.png', dpi=120, bbox_inches='tight')
plt.close()
print('\n已保存 chart_01.png — 24 小时市场重叠图')

# ============ 4. 最小变动价位对应的物理下限基点价差 ============
# 物理下限价差 = 2 * tick_size(买卖各报一档 BBO)
# 基点价差 = (2 * tick_size) / price * 10000
names_short = ['A 股(茅台)', '美股(苹果)', '港股(腾讯)', '微型标普', '加密(BTC)']
min_bps = []
for name, m in markets.items():
    bps = (2 * m['tick_size']) / m['rep_stock_price'] * 10000
    min_bps.append(bps)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(names_short, min_bps, color=colors[:len(min_bps)], alpha=0.8)
ax.set_ylabel('物理下限基点价差(bps)', fontsize=11)
ax.set_title('最小变动价位对应的物理下限基点价差 — 同标的同流动性下规则差异', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
for bar, bps in zip(bars, min_bps):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h*1.02, f'{bps:.2f} bps', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_02.png', dpi=120, bbox_inches='tight')
plt.close()
print('已保存 chart_02.png — 物理下限基点对比')

# ============ 5. 撮合规则差异对比(FIFO 对比 PRORATA) ============
# 假设同一价位有三家挂单 A 100手 / B 200手 / C 700手,新对手单 500手到达
queue = [('A', 100), ('B', 200), ('C', 700)]
new_order_qty = 500
total_qty = sum(q for _, q in queue)

# FIFO 先进先出
fifo_fills = {}
remaining = new_order_qty
for name, qty in queue:
    take = min(remaining, qty)
    fifo_fills[name] = take
    remaining -= take
    if remaining == 0: break
for name, _ in queue:
    if name not in fifo_fills: fifo_fills[name] = 0

# PRORATA 按比例分配
prorata_fills = {name: int(new_order_qty * qty / total_qty) for name, qty in queue}

print('\n撮合规则差异(同样 500 手对手单到达)')
print(f'{"客户":<8}{"挂单量":<12}{"FIFO 成交":<14}{"PRORATA 成交":<14}')
for name, qty in queue:
    print(f'{name:<8}{qty:<12}{fifo_fills[name]:<14}{prorata_fills[name]:<14}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
names = [n for n, _ in queue]
fifo_vals = [fifo_fills[n] for n in names]
prorata_vals = [prorata_fills[n] for n in names]
axes[0].bar(names, fifo_vals, color='#1d3557', alpha=0.8)
axes[0].set_title('FIFO 先进先出(美股 / A 股 / 港股 / 加密)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('成交量(手)')
for i, v in enumerate(fifo_vals):
    axes[0].text(i, v+15, str(v), ha='center', fontweight='bold')
axes[1].bar(names, prorata_vals, color='#e63946', alpha=0.8)
axes[1].set_title('PRORATA 按比例(芝商所利率期货)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('成交量(手)')
for i, v in enumerate(prorata_vals):
    axes[1].text(i, v+15, str(v), ha='center', fontweight='bold')
plt.suptitle('撮合规则差异 — 同样 500 手对手单到达三家挂单的分配', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_03.png', dpi=120, bbox_inches='tight')
plt.close()
print('已保存 chart_03.png — 撮合规则差异对比')
print('\n所有结果已生成。三张图分别展示:24h 重叠 / 物理下限基点 / 撮合规则差异')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 美股盘前数字陷阱 | 苹果 | 2023 年5 月某次苹果财报第2 天,盘前价格显示190 块,9:30钟一响真实开盘价跳到一百九十二,中间两块钱差距就是盘前那几个稀薄做市商报价跟真实开盘清算价的差距。盘前流动性只有日间的5%到10%,几个大单就能把价格推上去。散户教训 — 盘前价格只能当参考,真要抓开盘挂市价单别挂限价单。 |
| A 股涨停板封单陷阱 | 创业板个股 | 2023 年某创业板股票早上九点四十涨停,封单显示买盘3800000 股。某散户挂市价买1000 股,收盘零成交。第2 天该股票低开4.8%。涨停封单跟成交概率反相关 — 封单数量大不代表买盘真实强,反而经常接近顶部信号。这是 A 股最经典的散户陷阱之一。 |
| 沪深港通额度耗尽 | 腾讯港股 | 2023 年某周一上午通过北向额度购买腾讯港股,通道额度上午十一点已用80%,中午十二点用尽。下午腾讯走高3%但内地散户的挂单完全无法成交。沪深港通五百二十亿日额度在异常事件期间是真实瓶颈,大单提前两3 天预约或者拆成多日分批是基础操作。 |
| 2015 年8 月24 日全球闪崩 | 标普五百 / 沪深三百 / 比特币 | 同一天美股开盘5 分钟限价熔断触发一千多次但市场仍跌7%,A 股千 股跌停只挂卖单完全没买盘,加密市场比特币24 小时跌18%撮合一直跑无任何保护。三种保护机制完全不同的处理 — 美股是瞬间熔断挂单,A 股是连续跌停流动性消失,加密是裸奔到底。这是写跨市场策略最大的陷阱。 |
| 芝商所按比例分配策略迁移失败 | 欧洲美元利率期货 | 某机构2022 年将纳斯达克高频做市策略搬到芝商所欧洲美元利率期货,直接套用抢内侧加一档零点零零零一美元的策略,挂100 手在十万手对手单进来时只分到一手完全没法成交。重写按比例分配版本改成大单挂多档总量提到10000 手以上才跑起来。撮合规则差不只是参数差,是策略原理差。 |
| 2022 年5 月露娜币崩盘 | 露娜币 | 露娜币5 天跌幅99.99%,从一百多块跌到零点零零零几,撮合引擎正常运行价格一路向下没有任何熔断。这是加密市场没有保护伞最经典的案例,印证了加密策略必须自己实现单日亏损上限熔断这条基础底线。同期传统市场如果出现类似跌幅早就触发交易所级停盘,加密永远不会。 |


## 常见坑

### ⚠ 01. 把美股直觉直接套到 A 股

美股是 T 加零加多场所加限价熔断加9:30开盘连续撮合,A 股是 T 加一加单交易所加涨跌停加集合竞价开盘加九点二十之后不能撤单。3 条规则单独一条就能让美股策略失效。正确做法 — 跨市场策略必须把交易时段、午休、撮合、最小变动、涨跌停、熔断这六个参数做成可配置字典,不要硬编码在策略主体里。

### ⚠ 02. 把加密24 小时当成连续高流动性

加密成交量分布极不均匀,北美下午到亚洲晚上是高峰,周末整体只有平时的两三成,亚洲深夜到欧洲凌晨是低谷。你写算法挂大单不分时段,在低成交时段冲击成本会被放大好几倍。正确做法 — 加密策略要按时段加权流动性,低峰期减少挂单频率或者改用小批量分批。

### ⚠ 03. 涨停板看封单大就追

涨停板的封单数量跟成交概率反相关,封单越大越难成交,而且第2 天大概率低开。看到涨停板封单几百万股,十有八九是顶部信号,不是底部信号。正确做法 — 涨停板追涨是个高概率被套的坑,看到封单巨大的票按住手,等次日开盘真实成交结构出来再判断。

### ⚠ 04. 做沪深港通忽略额度和汇率

日额度月额度耗尽你的单挂不上,汇率波动让你哪怕股票不动也亏几个百分点,假日不同步让你看着港股动了也参与不了。正确做法 — 沪深港通大单提前盯紧通道使用率,人民币兑港币汇率对冲在外汇市场上做,假日日历提前一个月排好。

### ⚠ 05. 把芝商所所有期货当成按比例分配

芝商所只有部分利率期货和英镑国债期货是按比例,微型标普和大部分商品期货还是先进先出。新手以为芝商所一锅端按比例,策略就会用错排队位置策略。正确做法 — 任何新接触的期货合约第一步先确认它的撮合规则,芝商所官网每个合约页面都明确标注是 FIFO 还是 Pro-Rata。

### ⚠ 06. 加密大仓位长期放交易所

2022 年某交易所一周破产,2014 年门头沟比特币丢失,2023 年多家交易所遭监管打击限制提币。加密交易所有三个尾部风险 — 黑客攻击、跑路、监管打击。你大仓位长期放交易所就是裸奔。正确做法 — 短期交易仓位放交易所,中长期持仓转到硬件钱包或多签钱包,70%以上仓位绝不长期暴露在单一交易所。

## 实战 SOP · 跨市场实战7 条铁律

1. 学新市场前必读这个市场的交易规则手册,上交所、港交所、纳斯达克、芝商所、币安都公开,最多花一个小时通读一遍
2. 跨市场策略要把交易时段、午休、撮合规则、最小变动、涨跌停、熔断六个参数做成可配置字典,不同市场切换只改字典不改代码主体
3. 最小变动价位决定价差的物理下限,小盘股低价票要先算物理下限再做策略,港股按价格分档的低价票物理下限比 A 股大十几倍
4. 异常状态保护机制 A 股最严、美股中等、加密为零,跨市场策略要按市场单独配置异常处理逻辑,加密必须自己实现单日亏损上限
5. 沪深港通额度和汇率是看不见的成本,大单提前盯紧通道使用率,人民币兑港币汇率对冲在外汇市场上做
6. 涨停板封单跟成交概率反相关,不要看封单大小判断方向,要看资金流向和成交结构
7. 加密大仓位不要长期放交易所里,黑客跑路监管打击三个尾部风险随时可能让你血本无归,中长期持仓转硬件钱包是底线

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 全球四类主流市场气质完全不同 — 美股像咖啡店集中六个半小时,A 股像食堂4 小时加午休,港股像短工时食堂,加密像24 小时便利店
3. T 加一是 A 股最大的独特规则,逼着散户做趋势不做日内,止损执行有滞后,这是 A 股策略架构的核心约束
4. 异常保护机制三套完全不同 — A 股涨跌停最严,美股限价熔断中等,港股2016 年起加限价熔断,加密无任何保护必须自己实现停机
5. 撮合规则绝大多数市场用先进先出加价格优先,只有芝商所部分利率期货英镑国债用按比例分配,策略原理完全不同
6. 最小变动价位决定价差物理下限,港股按价格分档让低价票物理下限比 A 股大十几倍,不是流动性差是规则差
7. 沪深港通是 A 股加港股的官方通道,额度汇率假日三个隐性成本必须监控,异常事件下额度是真实瓶颈
8. 加密跨交易所价差走宽不是套利信号,是异常事件预警,看到价差拉到1%以上要警觉某家交易所可能出问题
9. 跨市场策略的核心思路是把市场参数做成配置化字典,而不是把一套美股直觉硬套到 A 股或加密上去交学费

## 自测题

**Q1.** A 股的 T 加一规则跟美股的 T 加零有什么实战差别?为什么 A 股很多散户被迫做趋势?  答:T 加一是当天买的股票第2 个交易日才能卖,T 加零是当天买当天能卖。A 股 T 加一直接导致 — 第一,日内回转策略做不了,你不能上午买下午卖;第二,止损执行有滞后,上午买了下午跌了你跑不掉;第三,逼着散户做趋势,因为必须持仓到第2 天才能卖,所以必须判断隔夜消息和大盘方向。美股 T 加零让你一天可以无限次交易,但有日内交易规则限制账户余额不足两万5000 美元的散户一周最多三次日内交易。

**Q2.** A 股涨停板的封单数量为什么跟成交概率反相关?为什么追涨停大概率被套?  答:涨停板上卖盘几乎没有,所有买单按时间排队等卖盘出现。封单几百万股意味着你排在很后面,前面的人都买不到你更买不到。第2 天涨停板大概率低开,原因是昨天封单里有相当一部分是没买到的散户跟风挂的虚单,看到次日没继续涨停心态崩了变成卖盘往外砸。所以涨停板封单大不是底部信号,反而经常是顶部信号。

**Q3.** 为什么把美股高频做市策略搬到芝商所欧洲美元利率期货会失效?  答:美股用先进先出加价格优先,同价排队第一位下一秒就能成交,所以高频拼速度抢内侧。芝商所欧洲美元利率期货用按比例分配,同一价位来对手单按各家挂单数量的比例分。你挂100 手别人挂10000 手新对手单进来你只能分到大约百分之一。两套规则下排队位置的重要性完全不同 — 先进先出拼速度,按比例分配拼挂单数量。策略原理不同不是参数差。

**Q4.** 沪深港通对内地散户有哪三个看不见的成本?  答:第一是汇率成本 — 港股报价用港币结算用人民币,汇率波动几个百分点会直接吃掉你的收益。第二是额度限制 — 北向日额度五百二十亿人民币,异常事件期间可能上午就用完。第三是假日不同步 — 内地放假香港正常开市的时候通道关闭,你看着港股走但动不了,反向也一样。

**Q5.** 加密市场跟传统市场最大的结构性区别是什么?这个区别决定了加密策略必须做什么?  答:最大的结构性区别是加密没有任何强制熔断或停盘机制,撮合24 小时连续运行,价格爱跌多少跌多少。2022 年露娜币5 天跌99.99%撮合一路正常运行。这个区别决定加密策略必须自己实现风控 — 单日亏损达到本金的5%自动停机,这是基础底线;大仓位转硬件钱包不长期暴露在单一交易所,这是仓位结构底线;价差走宽到1%以上立刻警觉某家交易所可能异常,这是预警底线。三道底线缺一不可。

把答案写下来,3 天后再回看。

## 下一节预告

**Day 026 · 数据来源大全** (Data Sources)

第26 课进入第二阶段第一节,讲股票数据怎么获取怎么清洗。前25 天讲的是金融工程理论,接下来要从理论走向工程实战。第26 天我们会把主流的数据接口全部实测一遍,看接口限频、数据缺失、价格异常、复权因子这些真实工程问题怎么处理。这是从学量化走向做量化的转折点。

## 推荐阅读

- Harris《Trading and Exchanges: Market Microstructure for Practitioners》(2003,Oxford)— 市场微结构经典教科书,跨市场撮合机制对比一章是入门首选,把 NYSE / NASDAQ / CME 三大体系讲透
- Foucault Pagano Roell《Market Liquidity: Theory Evidence and Policy》(2013,Oxford)— 全球主要交易所流动性结构对比的学术经典,涵盖订单簿透明度跨市场套利通道
- 证监会《沪港通业务实施办法》(2014)+ 港交所《互联互通投资者教育手册》(每年更新)— 沪深港通官方权威文档,额度规则汇率结算假日日历最准确的来源
- Bouchaud Bonart Donier Gould《Trades Quotes and Prices: Financial Markets Under the Microscope》(2018,Cambridge)— 高频微结构 + 跨市场实证,撮合规则按比例分配在芝商所的实证章节质量很高
- Patel《CME Group: The Exchange of Exchanges》(2019,Wiley)— 芝商所的成长史,从猪牛羊到全球利率期货之王,撮合规则演化跟交易所合并背景结合,可读性强
- Python 工具栈:pandas + matplotlib 是跨市场静态对比的标准组合;实际数据接口 yfinance 拉美股,akshare 拉 A 股和港股通,binance-python 拉加密,Tushare 拉国内期货;本课用静态字典做对比图,避免数据接口门槛